In [40]:
import pandas as pd 
df=pd.read_excel('OnlineRetail.xlsx')
print(df.head())

  InvoiceNo StockCode                          Description  Quantity  \
0    536365    85123A   WHITE HANGING HEART T-LIGHT HOLDER         6   
1    536365     71053                  WHITE METAL LANTERN         6   
2    536365    84406B       CREAM CUPID HEARTS COAT HANGER         8   
3    536365    84029G  KNITTED UNION FLAG HOT WATER BOTTLE         6   
4    536365    84029E       RED WOOLLY HOTTIE WHITE HEART.         6   

          InvoiceDate  UnitPrice  CustomerID         Country  
0 2010-12-01 08:26:00       2.55     17850.0  United Kingdom  
1 2010-12-01 08:26:00       3.39     17850.0  United Kingdom  
2 2010-12-01 08:26:00       2.75     17850.0  United Kingdom  
3 2010-12-01 08:26:00       3.39     17850.0  United Kingdom  
4 2010-12-01 08:26:00       3.39     17850.0  United Kingdom  


In [41]:
df.shape

(541909, 8)

In [42]:
print(df.columns.tolist())

['InvoiceNo', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'UnitPrice', 'CustomerID', 'Country']


In [43]:
print(df.isnull().sum())

InvoiceNo           0
StockCode           0
Description      1454
Quantity            0
InvoiceDate         0
UnitPrice           0
CustomerID     135080
Country             0
dtype: int64


In [46]:
df=df.dropna(subset=['CustomerID'])
df.shape

(406829, 8)

In [50]:
order_per_customer=df.groupby('CustomerID')['InvoiceDate'].apply(lambda x: x.dt.date.nunique())
print(order_per_customer.head())

CustomerID
12346.0    1
12347.0    7
12348.0    4
12349.0    1
12350.0    1
Name: InvoiceDate, dtype: int64


In [52]:
repeat_target=(order_per_customer>1).astype(int)
print(repeat_target.value_counts())

InvoiceDate
1    2991
0    1381
Name: count, dtype: int64


In [53]:
baseline=repeat_target.value_counts(normalize=True).max()
print("Baseline Accuracy",baseline)

Baseline Accuracy 0.6841262580054894


In [54]:
print(df.columns)

Index(['InvoiceNo', 'StockCode', 'Description', 'Quantity', 'InvoiceDate',
       'UnitPrice', 'CustomerID', 'Country'],
      dtype='str')


In [58]:
df['amount']=df['UnitPrice']*df['Quantity']
features=df.groupby('CustomerID').agg(
    totalspent=('amount','sum'),
  avg_order_value=('amount','mean'),
  total_items=('Quantity','sum'),
  num_transactions=('InvoiceNo','nunique')
)

In [61]:
data=features.join(repeat_target.rename('repeat_custumer'))
data=data.dropna()
data.shape

(4372, 5)

In [74]:
import pandas as pd
from sklearn.model_selection import train_test_split,cross_val_score
from sklearn.tree import DecisionTreeClassifier
x=data[['totalspent','avg_order_value','total_items',]]
y=data['repeat_custumer']
tree=DecisionTreeClassifier(max_depth=4,random_state=42)
cross_val_scores=cross_val_score(tree,x,y,cv=5)
print('baseline:68%')
print("crossvalidationscores",round(cross_val_scores.mean(),3))


baseline:68%
crossvalidationscores 0.815


In [75]:
x_train,x_test,y_train,y_test=train_test_split(x,y,test_size=0.2,random_state=42)
tree.fit(x_train,y_train)
predict=tree.predict(x_test)
results=x_test.copy()
results['Actual']=y_test
results['Predicted']=predict
wrong=results[results['Actual'] != results['Predicted']]
print(len(wrong))

148
